In [1]:
import sys
sys.path.insert(0, "../src")

from tb_report import adapt, generate_report_llm

1 - Single strong active-TB region, upper zone (patient's left)

In [2]:
tb_output = adapt(
    probabilities={"healthy": 0.05, "sick_non_tb": 0.12, "tb": 0.83},
    detections=[
        ((350, 40, 500, 180), "active_tb", 0.91), 
    ],
    image_size=(512, 512),
)

print(tb_output.model_dump_json(indent=2))

{
  "image_classification": {
    "predicted_label": "tb",
    "probabilities": {
      "healthy": 0.05,
      "sick_non_tb": 0.12,
      "tb": 0.83
    }
  },
  "regions": [
    {
      "type": "active_tb",
      "confidence_band": "high",
      "location": "upper_left"
    }
  ]
}


In [3]:
print(generate_report_llm(tb_output))

/home/ase/code/vqa/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 195/195 [00:00<00:00, 1119.26it/s]
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all g

The radiological analysis indicates a high probability (83%) of tuberculosis (TB) infection in the upper left lung region, with significant confidence in this finding. The likelihood of the patient being healthy or having non-TB pulmonary disease is considerably lower. This suggests active TB disease in the specified area, warranting further clinical investigation and treatment.


In [4]:
ambiguous_output = adapt(
    probabilities={"healthy": 0.15, "sick_non_tb": 0.35, "tb": 0.50},
    detections=[
        ((60,  30, 200, 150), "active_tb",  0.62),  # upper zone, patient's right, medium confidence
        ((280, 300, 420, 450), "latent_tb", 0.45),  # lower zone, patient's left, low confidence
    ],
    image_size=(512, 512),
)

print(ambiguous_output.model_dump_json(indent=2))

{
  "image_classification": {
    "predicted_label": "tb",
    "probabilities": {
      "healthy": 0.15,
      "sick_non_tb": 0.35,
      "tb": 0.5
    }
  },
  "regions": [
    {
      "type": "active_tb",
      "confidence_band": "medium",
      "location": "upper_right"
    },
    {
      "type": "latent_tb",
      "confidence_band": "low",
      "location": "lower_left"
    }
  ]
}


In [5]:
print(generate_report_llm(ambiguous_output))

[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The TB detection model has identified an active tuberculosis (TB) lesion in the upper right quadrant of the image, with a medium confidence level. Additionally, a latent TB presence with low confidence is noted in the lower left quadrant. The highest probability assigned is to an active TB infection (50%).


## Case 3 — Healthy, no detections

In [6]:
healthy_output = adapt(
    probabilities={"healthy": 0.91, "sick_non_tb": 0.07, "tb": 0.02},
    detections=[],
    image_size=(512, 512),
)

print(generate_report_llm(healthy_output))

[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The radiologist-reviewed image analysis indicates a high probability (91%) of the patient's lung images being classified as healthy with no signs of tuberculosis (TB), as the TB likelihood is minimal at 2%. No abnormal regions were identified in the report.
